In [0]:
# ---------------------------------------------------------------------------------------
# SCRIPT: 4.1_ceap_despesas_incremental
# LOCAL: 1_bronze/src/4_gastos_ceap/
# OBJETIVO: Gerar dados de despesas do CEAP incrementalmente fornecendo a tabela bronze_ceap_despesas
# ENTREGA: Ingestão incremental de /deputados/{id}/despesas com controle de paginação
# ---------------------------------------------------------------------------------------

import requests as req
import time

# Filtrando apenas deputados da legislatura atual (57)
df_mestre = spark.table("gold_atlas_frentes_parlamentares")
deputados_ativos = df_mestre.filter("id_legislatura = 57") \
                            .select("id_deputado").distinct().collect()

deputados_ids = [int(row['id_deputado']) for row in deputados_ativos]

despesas_totais = []
cont_sucesso = 0

# Ingestão: 10 deputados que possuam dados de despesas
for id_dep in deputados_ids:
    if cont_sucesso >= 10: break
    
    url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_dep}/despesas"
    params = {"ordem": "ASC", "ordenarPor": "ano", "itens": 100}
    
    try:
        res = req.get(url, params=params)
        if res.status_code == 200:
            dados_json = res.json()
            if dados_json['dados']:
                # Injetado o ID do deputado em cada registro de despesa
                for item in dados_json['dados']:
                    item['idDeputado_fix'] = id_dep
                
                despesas_totais.extend(dados_json['dados'])
                cont_sucesso += 1
                print(f"Deputado {id_dep}: {len(dados_json['dados'])} despesas colhidas.")
            else:
                print(f"Deputado {id_dep}: Sem despesas registradas.")
        else:
            print(f"Deputado {id_dep}: Status {res.status_code}")
            
    except Exception as e:
        print(f"Erro no ID {id_dep}: {e}")

# Salvar
if despesas_totais:
    df_ceap = spark.createDataFrame(despesas_totais)
    df_ceap.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_ceap_despesas")
    print(f"\n Total final: {len(despesas_totais)} registros de {cont_sucesso} deputados.")

In [0]:
display(spark.table("bronze_ceap_despesas"))